# *Solutions function which uses LU Decomposition instead of Ref {Gaussian Elimination}

In [ ]:
function solutions(U::Matrix{<:Number},b_raw::Matrix{<:Number},r::Int64)
    # PA =LU
    # A = P^T * LU
    # Ax = b = P^T * LUx
    # b = P^T * LUx
    # Pb = LUx
    # Pb = Ly
    # Ux = b
    # Ax = b : b = m , y = m 
    # LU = PA : A = m*n, P = m*m, L = m*m , U = m*n, THESE ARE DIMENSIONS
    
    rank = r
    sz = size(U)    

    particular_vector = zeros(Float64,sz[2],1)
    row_status = zeros(Int64,sz[1],2) 
    # rows of row_status = number of rows
    # first column tells wheather a row is pivot row(1) or free row(0) 
    # second column tells in which column that pivot is 
    column_status = zeros(Int64,sz[2],1)
    # columns of column_status = number of columns in A which is A|b = U
    # first row tells wheather a column is pivot column(1) or free column(0)
    
 @inbounds  for i in 1:1:sz[1]
    @inbounds     for j in 1:1:sz[2]
            if U[i,j] !=0
                row_status[i,1] = 1
                row_status[i,2] = j
                column_status[j] =1
                break
            
            end
        end
    end # this code block is used to find the pivot columns and pivot rows, and store them in row_status and column_status respectively

@inbounds for i in sz[1]:-1:1
        if row_status[i,1] ==0
             nothing
        else
            pivot_column = row_status[i,2]
            b_comp = y[i]
          @inbounds @simd  for doraemon in pivot_column+1:1:sz[2]
                if column_status[doraemon] == 0
                    nothing
                else
                    b_comp -= U[i,doraemon]*particular_vector[doraemon]
                end
            end
            particular_vector[pivot_column]= b_comp/U[i,pivot_column]
        end
    end  # this code block is used to find the particular solution of the system of linear equations, and store it in particular_vector

    if rank == sz[2]
        return particular_vector
    else 

       linear_combination_of_vectors = Matrix{Float64}[]   
@inbounds for j in 1:1:sz[2]
            if column_status[j] == 0 #= We only care about free columns where column_status[j] == 0
                                         Because the pivot variabels of basis vector are (-1)*M[:,j] where j is a free column
                                        =#
            
            basis_vector = zeros(Float64,sz[2],1)
            basis_vector[j] = 1 # the variable of free column(free variable is set to 1 => this is linear algebra theory)
          @inbounds for i in sz[1]:-1:1
                if row_status[i,1] ==0
                    nothing
                else
                    pivot_column = row_status[i,2]
                    b_comp = 0
                    @inbounds @simd  for doraemon in pivot_column+1:1:sz[2]
                        b_comp -= U[i,doraemon]*basis_vector[doraemon]
                                     end
                    basis_vector[pivot_column]= b_comp/U[i,pivot_column]
                end
            end
            push!(linear_combination_of_vectors,basis_vector)
        end
    end # this code block is used to find the linear combination of vectors of the system of linear equations, and store it in linear_combination_of_vectors


return particular_vector, linear_combination_of_vectors   
        
end
end

solutions (generic function with 1 method)

# A lot of LU decomposition is same as row echelon form and reduce row echelon form
# Hence in my unpartial pivoted code i have copied the ref code 
# And then i did slight changes in the ref code to make the unpartially pivoted LU code


In [4]:
# final LU with partial pivoting 
function LU(M :: Matrix{<:Number})
    U = Float64.(M)
    sz = size(U)
    L = zeros(Float64, sz[1], sz[1])
    P = zeros(Float64, sz[1], sz[1])
  @inbounds @simd  for modi in 1:1:sz[1]
        L[modi, modi] = 1
        P[modi, modi] = 1
    end
    i = 1
    rank = 0
    h = 1 
    
    while i <= sz[1] && h <= sz[2] # traversing both rows and columns
        pivot_row = i
        pivot = abs(U[i, h])
        
        # Scan ALL rows below row i in the current column h
        segment = @view U[i+1:sz[1], h]
    @inbounds @simd for meow in 1:1:sz[1]-i
            temp = abs(segment[meow]) 
            if temp > pivot # partial pivoting is occuring here
                pivot = temp
                pivot_row = meow + i
            end
        end
        
        # If the largest element in this column is 0 then skip to the next column
        # Because a number like 0 or near zero leads to large numbers ( when divided by 0 or near zero)
        # which leads to overflow and roundoff errors
        if pivot == 0
            h += 1
            continue
        end
        
        # pivot found hence increase the rank by 1
        rank += 1
        h_carry = h
        
        # rows of U swapped
        temp = U[i, :]
        U[i, :] = U[pivot_row, :]
        U[pivot_row, :] = temp
        
        # rows of P swapped
        temp = P[i, :]
        P[i, :] = P[pivot_row, :]
        P[pivot_row, :] = temp
        
        # rows of L swapped
        if i > 1
            temp = L[i, 1:i-1]
            L[i, 1:i-1] = L[pivot_row, 1:i-1]
            L[pivot_row, 1:i-1] = temp
        end 
        
        pivot_val = U[i, h_carry]
        @inbounds @simd for j in i+1:1:sz[1]
            mbp = U[j, h_carry] / pivot_val 
            L[j, i] = mbp
            @inbounds @simd for k = 1:1:sz[2]
                U[j, k] = U[j, k] - mbp * U[i, k]
            end
        end
        
        i += 1
        h += 1
    end
    return P, L, U, rank
end

LU (generic function with 1 method)

In [3]:
# attempt to make LU with partial pivoting
function LU(M :: Matrix{<:Number})
    U = Float64.(M)
    sz = size(U)
    L = zeros(Float64, sz[1], sz[1])
    P = zeros(Float64, sz[1], sz[1])
    for modi in 1:1:sz[1]
        L[modi, modi] = 1
        P[modi, modi] = 1
    end
    i = 1
    rank = 0
    while i <= sz[1]
        pivot = 0
        h_carry = 0
        h = 1
        while h <= sz[2]
            if U[i, h] == 0 
                h += 1
            else 
                rank += 1
                pivot_row = i
                pivot = U[i, h]
                h_carry = h
                segment = @view U[i+1:sz[1], h]
                for meow in 1:1:sz[1]-i
                    if segment[meow] == 0
                        nothing 
                    else
                        temp = abs(segment[meow])
                        if abs(pivot) >= temp
                            nothing
                        else
                            pivot = segment[meow] 
                            pivot_row = meow + i
                        end
                    end
                end
                # rows of U swaped
                temp = U[i, :]
                U[i, :] = U[pivot_row, :]
                U[pivot_row, :] = temp
                # rows of P swaped
                temp = P[i, :]
                P[i, :] = P[pivot_row, :]
                P[pivot_row, :] = temp
                # rows of L swaped
                if i > 1
                    temp = L[i, 1:i-1]
                    L[i, 1:i-1] = L[pivot_row, 1:i-1]
                    L[pivot_row, 1:i-1] = temp
                end 
                break 
            end
        end 
        if pivot == 0
            i += 1
            continue
        end
             
        @inbounds @simd for j in i+1:1:sz[1]
            mbp = U[j, h_carry] / pivot 
            L[j,i] = mbp
            @inbounds @simd for k = 1:1:sz[2]
                U[j, k] = U[j, k] - mbp * U[i, k]
            end
        end
        i += 1
    end
    return P, L, U,rank
end


LU (generic function with 1 method)

In [5]:
# LU function but not partially pivoted
function LU(M :: Matrix{<:Number})
    U = Float64.(M)
    sz =size(U)
    L  = zeros(Float64,sz[1],sz[1])
    P  = zeros(Float64,sz[1],sz[1])
    for modi in 1:1sz[1]
        L[modi,modi] = 1
        P[modi,modi] = 1
    end
    i =1
    rank = 0
 while i<= sz[1] # Selects/locks the rows(selection of pivot row), The pivot is selected in this loop only
            pivot = 0
            h_carry = 0
            h =1
  while h<=sz[2]
                if U[i,h] != 0
                    pivot = U[i,h]
                    h_carry = h
                    rank+=1
                    break
                else
                     meow = i+1
                    if meow <= sz[1]
                            while meow<= sz[1]
                            if  U[meow,h] != 0
                            
                                temp = U[meow, :]
                                U[meow, :] = U[i, :]
                                U[i, :] = temp
                               #----------------------
                               # L[h,1:h-1],L[h+1,1:h-1] =L[h+1,1:-h] ,L[h,1:h-1] 

                                temp = L[h+1,1:h-1]
                                L[h+1,1:h-1] = L[h,1:h-1]
                               L[h,1:h-1] = temp
                                    
                                #----------------------

                                temp = P[meow, :]
                                    P[meow, :] = P[i, :]
                                    P[i, :] = temp
                                break                                   
                            else
                                if meow != sz[1]
                                    meow+=1
                                else # meow == sz[1]
                                    h+=1
                                    break
                                end
                            end
                        end
                    else
                        h+=1
                    end
                end
            end  
            if pivot == 0
                i+=1
                continue
            end
                 
@inbounds @simd  for j in i+1:1:sz[1] # its selects rows on which row operation is about to occur, multiplier is also selected in this loop
                mbp = U[j,h_carry]/pivot # mbp = multiplier by pivot i.e multiplier/pivot
                L[j,h_carry] = mbp
@inbounds @simd for k = 1:1:sz[2]
                    U[j,k] = U[j,k]-mbp*U[i,k]
                end
            end
            i+=1
    end
    return P,L,U
end

LU (generic function with 1 method)